# 01 — Pipeline-ul zilnic de știri financiare

**Echipa:** Tofan Bogdan, Manea Alina-Alexandra, Burcă Alina  
**Materia:** NLP

Acest notebook reproduce flow-ul `Daily news agent` din N8N ca proiect Python rulabil pe Google Colab.

## Ce face

1. Preia știri din **NewsData.io** (politică, economie, Bitcoin, Ethereum, ecosistem Solana) — 5 surse în paralel.
2. Face **scraping** pe linkul fiecărui articol și curăță HTML-ul.
3. Folosește un **Information Extractor (DeepSeek)** ca să extragă doar conținutul util.
4. Clasifică sentimentul cu **FinBERT** (HuggingFace Inference API).
5. Agregă metadatele și împarte articolele în **Bullish / Neutral / Bearish**.
6. Trei **sub-agenți DeepSeek** sintetizează fiecare categorie.
7. Un **agent principal Grok** combină cele 3 brief-uri și interoghează **Pinecone** pentru paralele istorice.
8. Raportul final + audio Opus (OpenAI TTS) sunt trimise pe **Telegram**.

Tot ce vezi mai jos este implementat în pachetul `src/` — celulele de aici doar orchestrează.

In [ ]:
%cd /content
!git clone -b Fin-news-agent-dev https://github.com/BogdanT54/financial-news-agent.git
%cd /content/financial-news-agent
!pip install -q -r requirements.txt

In [ ]:
import sys, os

PROJECT_ROOT = '/content/financial-news-agent'
if os.path.isdir(PROJECT_ROOT) and PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)
elif os.path.basename(os.getcwd()) == 'notebooks':
    sys.path.insert(0, os.path.abspath('..'))

# Încarcă secretele din Colab Secrets → os.environ ÎNAINTE de a crea Settings
from src.config import load_colab_secrets, get_settings
print('--- Colab Secrets ---')
load_colab_secrets(verbose=True)

# Creează settings DUPĂ ce secretele sunt în os.environ
settings = get_settings()
print()
print('Pinecone index :', settings.PINECONE_INDEX)
print('MongoDB        :', settings.MONGO_DB)
print('Sub-agent model:', settings.MODEL_SUB_AGENT)
print('Main model     :', settings.MODEL_MAIN_AGENT)
print('DRY_RUN        :', settings.DRY_RUN)

## 2. Verificare credentials

Verifică rapid că ai completat cheile esențiale în `.env`.

In [ ]:
# settings e deja creat în celula de mai sus cu secretele încărcate
required = {
    "NewsData.io": settings.NEWSDATA_API_KEY,
    "HuggingFace": settings.HF_API_TOKEN,
    "OpenRouter":  settings.OPENROUTER_API_KEY,
    "OpenAI":      settings.OPENAI_API_KEY,
    "Pinecone":    settings.PINECONE_API_KEY,
    "MongoDB":     settings.MONGO_URI,
    "Telegram":    settings.TELEGRAM_BOT_TOKEN,
}

all_ok = True
for name, val in required.items():
    icon = '✅' if val else '❌'
    if not val:
        all_ok = False
    print(f'  {icon} {name:<12}: {"OK" if val else "LIPSEȘTE"}')

print()
if all_ok:
    print('✅ Toate cheile sunt OK — poți continua.')
else:
    print('⚠️  Adaugă cheile lipsă în Colab Secrets (🔑 din stânga) și activează Notebook access.')

## 3. Pas cu pas: fetch știri

Cele 5 fetchere rulează în paralel cu `ThreadPoolExecutor`.

In [ ]:
from src.news_fetcher import fetch_all_news, merge_news_sources

responses = fetch_all_news()
for source, payload in responses.items():
    n = len(payload.get("results") or [])
    print(f"  {source:<10}: {n} articole")

raw_articles = merge_news_sources(responses)
print(f"\nTotal după merge: {len(raw_articles)} articole")

## 4. Pas cu pas: scraping + clean + extract pentru un articol

Demonstrăm întregul lanț pe un singur articol înainte să rulăm pe toate.

In [ ]:
from src.scraper import fetch_article_html, clean_html, extract_main_content

sample = raw_articles[0]
print("Title:", sample.get("title"))
print("Link :", sample.get("link"))

raw_html = fetch_article_html(sample["link"])
print(f"\nHTML brut: {len(raw_html)} caractere")

clean = clean_html(raw_html)
print(f"După clean: {len(clean)} caractere")
print("\nPrimele 500 caractere:\n", clean[:500])

In [ ]:
extracted = extract_main_content(clean)
print("Conținut extras (primele 800 caractere):\n")
print(extracted[:800])

## 5. Pas cu pas: clasificare FinBERT

Apelăm modelul `ProsusAI/finbert` prin HuggingFace Inference API.

In [ ]:
from src.finbert import classify_sentiment

predictions = classify_sentiment(
    sample.get("title", ""),
    sample.get("description", ""),
    extracted,
)
for p in predictions:
    print(f"  {p['label']:<10}: {p['score']:.4f}")

## 6. Pas cu pas: agregare metadate

Funcția `build_article_record` întoarce un dict cu exact aceleași câmpuri ca nodul `Aggregate data` din N8N.

In [ ]:
from src.aggregator import build_article_record

record = build_article_record(sample, extracted, predictions)
for k in ["title", "sentiment", "confidence_level", "finbert_score",
         "score_positive", "score_neutral", "score_negative", "coins"]:
    print(f"  {k:<18}: {record[k]}")

## 7. Demo formattere (Bullish / Neutral / Bearish)

Folosim un dataset mic mock ca să vedem cum arată blocurile colorate.

In [ ]:
from src.formatters import format_bullish_block

mock = [
    {**record, "sentiment": "Bullish", "finbert_label": "positive",
     "finbert_score": 0.92, "confidence_level": "High",
     "score_positive": 0.92, "score_neutral": 0.05, "score_negative": 0.03}
]
print(format_bullish_block(mock))

## 8. Rulare pipeline complet (end-to-end)

`run_daily_pipeline()` execută tot lanțul:
- **`persist=True`** → scrie articolele în **MongoDB** + **Pinecone** (poți vedea live)
- **`send_telegram=False`** → nu trimite pe Telegram
- **`max_articles=5`** → limitează la 5 articole pentru demo rapid

Pentru a trimite și pe Telegram, schimbă `send_telegram=True`.

In [ ]:
from src.pipeline import run_daily_pipeline

# persist=True      → scrie în MongoDB + Pinecone
# send_telegram=False → NU trimite pe Telegram
result = run_daily_pipeline(max_articles=5, persist=True, send_telegram=False)
print("\n=== STATS ===")
for k, v in result["stats"].items():
    print(f"  {k:<22}: {v}")

### Raportul final

In [ ]:
from IPython.display import Markdown, Audio, display

display(Markdown(result["report"]))

### Audio raport (OpenAI TTS Opus)

In [ ]:
if result["audio_bytes"]:
    display(Audio(data=result["audio_bytes"], autoplay=False))
else:
    print("Audio neindisponibil (TTS a eșuat sau e dezactivat).")

## 9. Sumar persistență — ce s-a scris în MongoDB + Pinecone

După rularea pipeline-ului cu `persist=True`, verificăm ce a ajuns în baza de date și vector store.

In [ ]:
from src.mongo_store import count_articles, get_recent_articles

total = count_articles()
n_added = result['stats']['articles_processed']
recent = get_recent_articles(n=n_added)

print(f'📦 MongoDB — colecția "{settings.MONGO_ARTICLES_COLLECTION}"')
print(f'   Total articole în colecție : {total}')
print(f'   Adăugate în această rulare : {n_added}')
print(f'\nUltimele {n_added} articole inserate:')
print('─' * 90)
for d in recent:
    sent = d.get('sentiment', '?')
    icon = {'Bullish':'🟢', 'Bearish':'🔴', 'Neutral':'⚪️'}.get(sent, '⚫️')
    score = d.get('finbert_score', 0)
    title = (d.get('title', '') or '')[:70]
    src = d.get('source_name', '')
    print(f'  {icon} {sent:<8} | score={score:.2f} | {title}')
    print(f'      source: {src}')
print('─' * 90)

In [ ]:
from src.vectorstore import get_index_stats, init_pinecone

stats = get_index_stats()
print(f'🌲 Pinecone — index "{settings.PINECONE_INDEX}"')
print(f"   Total vectori în index : {stats.get('total_vector_count', 'N/A')}")
print(f"   Dimensiune embedding    : {stats.get('dimension', 'N/A')}")
print()

# Verificăm că putem regăsi articolele tocmai inserate prin similarity search
first_article = result['processed_articles'][0]
query = first_article['title']
print(f'🔍 Test similarity search cu query:')
print(f'   "{query[:80]}"')
print()
store = init_pinecone()
results = store.similarity_search(query, k=3)
for i, doc in enumerate(results, 1):
    m = doc.metadata
    print(f'  [{i}] {m.get("title", "")[:70]}')
    print(f'      label={m.get("finbert_label", "?")} | score={m.get("finbert_score", 0):.2f} | {m.get("publishing_date", "")[:10]}')

## 10. Mod live (full live) (Telegram + persistență)

Pentru o rulare reală (mesaj pe Telegram + scriere în Pinecone/Mongo), schimbă în `.env`:

```
DRY_RUN=false
```

Sau forțează din cod cu argumentele funcției:

In [ ]:
# Rulare LIVE — scrie în Pinecone+Mongo și trimite pe Telegram
# result_live = run_daily_pipeline(max_articles=10, dry_run=False)